# E6 — GeoStructureNet masked three-way · Full Colab pipeline → **NEW submission**

**One path:** cached 10 m DEM → 100 m geological channels → actual E6 three-way GPU training (3 geographic CV folds + 3 withheld-system splits) → pooled DTI → 3-fold full-region inference → `submission_e6_geostructurenet.tif` → manual DrivenData upload.

**This notebook never uses the old W2 TIFF as the submission.** The existing terrain / potential-field encoders start from the genuine E1 checkpoint of the **same fold** when available; the missing fold 2 and withheld-system splits train from scratch to prevent label leakage. The DEM mask is a separate gating constraint, not an input signal. Uncovered pixels use the two-way baseline logits exactly.

Data efficiency: read *only* the already saved 10 m G07 coarse raster parts; do not download new 1 m DEMs, execute 2 m feature extraction, train E1/E2/E3/E4/E5 comparisons, mirror raw data, run unit tests, generate research ablations or export old submissions. Intermediate tiles, per-feature caches, per-epoch checkpoints and per-fold predictions are saved and reused. If Colab disconnects, rerun the same notebook and continue.


In [ ]:
# 00 — Parameters (no hidden branches)
REPO = "mybenkhadda/GEMS---drivendata"
REF = "e6-submission-fasttrack"
CODE = "/content/GEMS_E6"
LOCAL = "/content/gems_work"
DRIVE = "/content/drive/MyDrive/GEMS_2"
EXPERIMENT_FILE = "configs/experiments/e6_masked_threeway_full.yaml"
RUN_TRAINING = True
RUN_EXPORT = True


## 01 — Mount, checkout and stage the **minimum** data

Git clone contains source only. Stage necessary original GeoDAWN inputs, folds and masks, not the complete historical 7 GB feature tree or original USGS DEMs. Feature caches are loaded lazily by exact channel name and only when used. The source version is pinned by checkout. No tests are invoked.


In [ ]:
import os, sys, subprocess
from pathlib import Path
from google.colab import drive
from IPython.display import display, JSON

drive.mount("/content/drive")
assert Path(DRIVE).is_dir(), DRIVE
if not (Path(CODE) / ".git").exists():
    subprocess.run(["git", "clone", "--depth", "1", "--branch", REF,
                    "https://github.com/" + REPO + ".git", CODE], check=True)
else:
    subprocess.run(["git", "-C", CODE, "fetch", "--depth", "1", "origin", REF], check=True)
    subprocess.run(["git", "-C", CODE, "checkout", "--detach", "FETCH_HEAD"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", CODE + "[colab]"], check=True)
if CODE not in sys.path:
    sys.path.insert(0, CODE)
from gems_bench.runtime import colab
colab.configure(local_root=LOCAL, drive_root=DRIVE, code_root=CODE)
from gems_bench.artifacts.sync import DriveSync
s = DriveSync(Path(LOCAL), Path(DRIVE))
minimum = [
    "data/raw/geodawn", "data/raw/faults",
    "outputs/benchmark/folds", "outputs/benchmark/datasets",
    "outputs/eda/datasets", "outputs/eda/manifest.json",
]
needed = [f for f in minimum if (Path(DRIVE) / f).exists()]
print("[E6] Staging only necessary core inputs:", needed)
s.stage_in(needed)
HEAD = subprocess.check_output(["git", "-C", CODE, "rev-parse", "HEAD"], text=True).strip()
print("[E6] Code commit:", HEAD)


## 02 — Prepare cached 100 m E6 feature bank

Build or reuse the seven compact DEM structural proxies from the **existing** 10 m coarse parts. No native 1 m/2 m work. The existing magnetic/gravity channels and already cached derivatives stay unchanged in the potential-field branch. Any missing historical channels are staged lazily, not recomputed en masse. All generated DEM tiles, TIFF and channel caches receive deterministic cache paths.


In [ ]:
from gems_bench.experiment import Experiment
CONFIG_PATH = str(Path(CODE) / EXPERIMENT_FILE)
EXP = Experiment.from_yaml(CONFIG_PATH, root=LOCAL, drive=DRIVE, code=CODE)
print("[E6] Experiment ID:", EXP.id)
print("[E6] Model:", EXP.config.get("model.type"))
print("[E6] Preparing input cache, model and checkpoints...")
PREP = EXP.prepare()
print("[E6] Prepare result:", PREP)
print("[E6] Cached DEM features:", EXP.runner._e6_inputs_ready)


## 03 — TRAIN E6 for real (resumable GPU)

Three geographic folds and three withheld-system splits, *not* a quick screen. Architecture: gated E1 terrain/field baseline + smaller DEM encoder + log-coverage three-way softmax; modality dropout 30%. Warm-start E1 CV fold 0 and 1 by their own checkpoints; missing fold 2 starts untrained for isolation. E6 checkpoints persist each epoch and each fold/split to Drive. Frozen baseline for the first two epochs where a warm-start exists; fine-tune afterward with a reduced baseline learning rate.

Default budget: ten epochs, 36 batches per epoch, FP16/BF16 GPU acceleration where supported. Adjust `configs/experiments/e6_masked_threeway_full.yaml` **before the first run** if a different training budget is required; changing the semantic config creates a new experiment ID rather than reusing an incompatible checkpoint.


In [ ]:
if RUN_TRAINING:
    print("[E6] Running complete CV + discovery: resumed units skipped automatically")
    TRAIN = EXP.train()
    print("[E6] Train complete:", TRAIN)
else:
    print("[E6] Training switch disabled; export requires all trained fold checkpoints")


## 04 — Score *only E6* on geographic and withheld-system splits

No W2 comparison, no ablation ladder, no threshold/weight searches. Saves `metrics/e6_dti.csv`, including fold and pooled competition-style DTI. The hidden leaderboard score cannot be known from local validation.


In [ ]:
EVAL = EXP.evaluate()
print("[E6] Evaluation:", EVAL)
if EXP.rd.has_table("e6_dti"):
    display(EXP.rd.load_table("e6_dti"))


## 05 — Full-region E6 inference and a NEW competition GeoTIFF

Checkpointed inference calculates three full-fold score rasters and caches each as soon as it completes. The final prediction is an average of the trained E6 models (not W2). One fixed, predeclared 1% eligible-area selection is made; the file is validated against the actual competition template: one float32 band, EPSG:32611, 100 m, correct NaN mask, values in [0, 1]. It is synced to Google Drive. No tests, no alternative model comparisons, no mirror of input DEMs.


In [ ]:
if RUN_EXPORT:
    OUTPUT = EXP.export()
    print("[E6] Submission status:", OUTPUT["status"])
    print("[E6] New E6 TIFF:", OUTPUT["path"])
    print("[E6] SHA256:", OUTPUT["sha256"])
    display(JSON(OUTPUT))
else:
    print("[E6] Export disabled (no substitute old W2 submission will be provided).")


## 06 — Download the new E6 submission for manual upload

Only the newly trained E6 TIFF is eligible. If model training or validation failed, this cell stops; no old champion substitution. The DrivenData competition currently allows **one scored submission per UTC day** and encourages appropriately licensed external data; preserve the G07/USGS source manifest and processing code for reproducibility.


In [ ]:
if not RUN_EXPORT or OUTPUT.get("status") != "READY_FOR_MANUAL_UPLOAD":
    raise RuntimeError("No new E6 submission is ready; finish training/export first")
from pathlib import Path
source = Path(OUTPUT["path"])
assert source.name.startswith("submission_e6_") and source.is_file(), source
print("Upload this NEW E6 result to DrivenData:", source)
from google.colab import files
files.download(str(source))
